# Molecular dynamics of small-molecule mixtures

This notebook runs a classical molecular dynamics simulation of water, ethanol,
oxygen and/or hydrogen in a periodic cubic box, then shows what happened: the
trajectory in VMD, the energies, the temperature, radial distribution functions
(RDFs) and bond phase-space diagrams. Units throughout: nm, ps, amu, kJ/mol and
K.

**Setup.** Install the package once, from the repository root, with
`pip install -e .[notebook]`, then start Jupyter (`jupyter lab`). The
[docs/internals.md](../docs/internals.md) documents every setting, the force field and the
modelling approximations to keep in mind when interpreting the results.

In [ ]:
from pprint import pprint

from jostle import Config, Results, check_config, find_vmd, run

## 1. Choose the settings

All settings live in a `Config`, in three groups: `system` (what is simulated),
`numerics` (how it is approximated) and `recording` (what is measured), plus
`end_time`, `seed` and `output_dir`. Any setting can be given by name, whatever
its group. [docs/internals.md](../docs/internals.md#2-configuration) lists them all with their
defaults.

The cell below runs a small box: 2 nm of water and ethanol for 5 ps, which
takes a few minutes. Delete `box_length` and `end_time` to run the defaults
instead (a 4 nm box for 10 ps, which takes about 40 minutes), or uncomment
and change other settings. The default `dt` (0.25 fs) resolves the fast O–H
and C–H vibrations; at 1 fs the energy is noticeably less well conserved.
Each measurement has its own interval: energies every `energy_interval` (20),
trajectory frames every `trajectory_interval` (100) and RDFs every `rdf_interval`
(100). The phase space is recorded every step, but only over the last
`phase_space_window` (1 ps) of the run.

`run` checks the settings before it starts (so does `config.replace`, which
returns a changed copy). It then prints the expected size of the output files,
and warns about settings that would waste the run or a measurement (a `dt`
too large for the bond vibrations, an RDF recorded too often, a likely typo in
a bond or pair name, ...); `check_config(config)` shows the same without running.
Results are written to `output_dir`, in a folder named after the settings.

In [ ]:
config = Config(
    dt = 0.00025,
    trajectory_interval = 1,
    box_length=2.0,                           # box edge (nm)
    end_time=0.5,                          # simulated time (ps)
    seed=1,                                # None: a different box each run
    rdf_bin_width=0.01,                           # RDF bin width (nm)
    # weights={"Water": 3, "Ethanol": 1},  # relative amounts of molecules
    # thermostat=False,                  # NVE instead of a thermostat
    # integrator="beeman",                     # or euler, verlet, velocity_verlet
    # trajectory_interval=100,                # steps between trajectory frames
    # phase_space_window=1.0,                # ps of phase space, at the end
    # rdf_pairs=["O~O (Water~Water)"],     # RDF pairs to record
    output_dir="../trajectories",
).validate()
pprint(config.to_dict(), sort_dicts=False)

## 2. Run the simulation

`run` builds and packs the box, integrates, and writes the trajectory and the
recorded data to the run folder. It returns a `Results` object with all the
recorded data. It makes no figures: those are made below, when you ask for
them.

To come back to a finished run later (after restarting the notebook, say),
load it instead of running again; everything below works on it:

```python
results = Results.load("../trajectories/<system>/<run>")
```

In [ ]:
results = run(config)

## 3. Watch the trajectory in VMD

The trajectory is `results.trajectory`: an `.xyz` file in Å, readable by
[VMD](https://www.ks.uiuc.edu/Research/vmd/) and similar tools. VMD is
optional. `view_in_vmd()` opens it in its own window, so the notebook stays
usable: the box outlined in red, surrounded by its 26 periodic images
(`show_periodic_images=False` leaves them out, which is lighter for large boxes). The
[docs/internals.md](../docs/internals.md#5-viewing-in-vmd) explains where jostle looks for VMD and
how to point it there (`vmd_path=...` or the `JOSTLE_VMD` environment variable).

The script VMD runs is kept in the run folder (`results.vmd_script`), so a
finished run can be reopened without Python: `vmd -e view_in_vmd.tcl` in that
folder.

In [ ]:
if find_vmd() is not None:
    results.view_in_vmd(show_box=False, show_image_outlines=False)
else:
    print("VMD not found; see docs/internals.md for how to point jostle to it.")

## 4. Plot the measurements

Each `plot_*` method returns a matplotlib figure, which is shown below the
cell. Nothing is saved unless you ask: `save=True` saves the figure in the run
folder (under a standard name), and `save="path/name.pdf"` saves it anywhere,
in any format matplotlib knows.

In [ ]:
results.plot_energy()

In [ ]:
results.plot_temperature()

### Radial distribution functions

An animation of the running mean of g(r) of each recorded pair over time. One
pair can be shown with its 90% confidence interval instead (e.g.
`results.animate_rdf("O~O (Water~Water)")`); the interval comes from blocks of
`block_time` (default 1 ps), so it appears once two blocks are complete. It
takes a few seconds to make.

In [ ]:
results.animate_rdf(max_frames=50)

### Phase space

Bond length against its rate of change, for a bond type named as in
`phase_space_bonds` (bond types absent from the box are skipped; without a
name, you get a figure for each tracked type). It shows the last
`phase_space_window` of the run, recorded every step, for up to 100 bonds of
each type spread through the box. This one is also saved, in the run folder's
`phase_space/`:

In [ ]:
results.plot_phase_space("O-H (Water)", save=True)

Where the computation time went:

In [ ]:
results.plot_timings()